# **01 - Load and Clean Data**

## Objectives

* Analyse online retail transaction data to understand customer behaviour, identify popular products, and optimise pricing and marketing strategies.

## Inputs

* data/source/online_retail.csv

## Outputs

* data/processed/online_retail_cleaned.csv

## Additional Comments

* Data sourced from Kaggle
* Covers transactions from a UK-based online retailer
* Cleaned data is used as input in 02_analysis_and_visualizations.ipynb



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os

current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_1/DA_project_1/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_1/DA_project_1'

## Install Dependencies

This cell installs all required packages listed in `requirements.txt`. 

In [ ]:
!pip install -r requirements.txt

# Import Libraries

In [4]:
import pandas as pd

# Load Data

In this section, we load the raw transaction data from the source CSV file and prepare it as a DataFrame for analysis.

* Input: data/source/online_retail.csv
* Output: A raw DataFrame (df_raw) containing all transactions

The shape is printed to confirm the number of rows and columns before any 
cleaning is applied.

In [5]:
df = pd.read_csv("data/source/online_retail.csv")
print(df.shape)

(541909, 8)


---

## Data Cleaning

### Handle Missing Values

In [6]:
df.isna().sum()

InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
dtype: int64

The output above shows the number of missing values per column. 
`Description` has 1,454 missing values. Since product name is a core 
identifier for our analysis and not the description, we dropped these rows.

In [7]:
df.dropna(subset=["Description"], inplace=True)
print(df.shape)

(540455, 8)


### Remove Duplicates

In [8]:
print(df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(by="InvoiceNo")

5268


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
...,...,...,...,...,...,...,...,...
440149,C574510,22360,GLASS JAR ENGLISH CONFECTIONERY,-1,2011-11-04 13:25:00,2.95,15110,United Kingdom
461407,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838,United Kingdom
461408,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838,United Kingdom
529980,C580764,22667,RECIPE BOX RETROSPOT,-12,2011-12-06 10:38:00,2.95,14562,United Kingdom


First we check how many duplicate rows exist, then inspect them before 
removing. Duplicates are removed entirely as they represent identical 
transactions likely caused by data entry errors rather than genuine 
repeat purchases.

In [9]:
df = df.drop_duplicates()
print(df.shape)

(535187, 8)


### Filter Invalid Transactions

Rows with a `UnitPrice` of zero or below are removed as they are not 
valid sales transactions — they likely represent test entries or data 
errors. Similarly, rows with negative or zero `Quantity` represent 
returns or cancellations and are excluded to focus the analysis on 
completed purchases only.

In [10]:
df = df[df["UnitPrice"] > 0]
print(df.shape)

(534129, 8)


In [11]:
df = df[df["Quantity"] > 0]
print(df.shape)

(524878, 8)


### Feature Engineering

A new column `TotalPrice` is created by multiplying `UnitPrice` by 
`Quantity`. This gives the total value of each transaction and will 
be used in the revenue analysis in notebook 02.

`InvoiceDate` is converted from a string to a datetime object, and 
`Month` and `Year` columns are extracted to enable time-based analysis.

In [12]:
df["TotalPrice"] = df["UnitPrice"] * df["Quantity"]
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [13]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Month"] = df["InvoiceDate"].dt.month
df["Year"] = df["InvoiceDate"].dt.year

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Month,Year
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,12,2010
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,12,2010
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010


In [3]:
# Confirm shape and preview new columns TotalPrice, Month and Year
print(df.shape)
df.head()

NameError: name 'df' is not defined

---

## AI Integration Note

GitHub Copilot was asked to review the data cleaning steps in this 
notebook. Copilot suggested the following improvements, which were 
implemented and had the effects described below:

- **Convert identifier columns to string:** Ensured consistent data 
  types for `InvoiceNo`, `StockCode` and `CustomerID`.
- **Remove cancellation invoices** (InvoiceNo starting with 'C'): 
  Filtered out additional rows not caught by the earlier Quantity filter.
- **Verify no negative TotalPrice remains:** The final check confirmed 
  that no negative values remained after cleaning: output was 0.

In [15]:
# Convert identifier columns to string (Copilot suggestion)
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)
df["CustomerID"] = df["CustomerID"].astype(str)

# Remove cancellation invoices (Copilot suggestion)
df = df[~df["InvoiceNo"].str.startswith("C", na=False)]
print("Rows after removing cancellations:", len(df))

# Verify no negative TotalPrice remains (Copilot suggestion)
print("Negative totals:", (df["TotalPrice"] < 0).sum())

print(df.shape)
print(df.isna().sum())
df.head()

Rows after removing cancellations: 524878
Negative totals: 0
(524878, 11)
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
Month          0
Year           0
dtype: int64


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Month,Year
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,12,2010
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,12,2010
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010


The cleaned DataFrame is saved to `data/processed/online_retail_cleaned.csv` 
for use in notebook 02. The file is then reloaded and previewed to confirm 
it was saved correctly.

In [16]:
df.to_csv("data/processed/online_retail_cleaned.csv", index=False)

In [17]:
df_cleaned = pd.read_csv("data/processed/online_retail_cleaned.csv")
print(df_cleaned.shape)
df_cleaned.head()

(524878, 11)


/var/folders/c0/b3t2bq595gxctk3zh1z0tr_h0000gn/T/ipykernel_97079/3109179276.py:1: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cleaned = pd.read_csv("data/processed/online_retail_cleaned.csv")


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Month,Year
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,12,2010
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,12,2010
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010


_____

## Data Handling Decisions

The following decisions were made during the cleaning process and the 
rationale for each is provided below:

- **Missing Description:** Rows with missing `Description` were dropped 
  rather than imputed, as product name is a core identifier that cannot 
  be meaningfully inferred from other columns.
- **Duplicate rows:** Removed entirely rather than aggregated, as they 
  represent identical transactions likely caused by data entry errors 
  rather than genuine repeat purchases.
- **Returns and cancellations:** Excluded (negative Quantity and 
  InvoiceNo starting with 'C') to focus the analysis on completed 
  purchases only. A separate export is recommended if refund behaviour 
  needs to be analysed.
- **Non-positive UnitPrice:** Removed as prices of zero or below are 
  not valid for a sales analysis and likely represent test entries or 
  data errors.

### AI-Generated Summary of Cleaning Steps

> **Note:** GitHub Copilot was asked to summarise the cleaning steps 
> performed in this notebook. The summary below is based on Copilot's output.

The following cleaning steps were performed in order:

1. Loaded raw CSV and inspected missing values
2. Dropped rows with missing Description (1,454 rows removed)
3. Removed exact duplicate rows (5,268 rows removed)
4. Filtered out non-positive UnitPrice values
5. Filtered out negative/zero Quantity values (returns)
6. Created TotalPrice column (UnitPrice × Quantity)
7. Converted InvoiceDate to datetime and extracted Month and Year
8. Converted identifier columns to string type
9. Removed cancellation invoices (InvoiceNo starting with 'C')
10. Saved cleaned dataset to data/processed/online_retail_cleaned.csv